# Download Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/download_data.ipynb)

Dataset page: [bobleesj/quantem-data](https://huggingface.co/datasets/bobleesj/quantem-data)

`quantem.data` provides small, public electron microscopy datasets hosted on Hugging Face. Use it as the friendly dataset layer: browse the registry, load a named dataset, and pass the returned `Dataset2d` or `Dataset3d` object directly into a widget.

This tutorial uses a real HAADF image of gold nanoparticles. The source array is 4096 by 4096 `uint16` data, so the notebook uses a calibrated preview crop for the interactive documentation page while keeping the full cached file available locally.

In [1]:
import numpy as np
import quantem.data as qdata

from quantem.core.datastructures import Dataset2d, Dataset3d
from quantem.widget import Show2D, Show3D

## Find a dataset

Use `qdata.tree()` for a compact text listing and `qdata.browse()` for a thumbnail picker. Both are safe for notebooks: they inspect dataset names, sizes, and tiny thumbnails without downloading multi-GB data.

In [2]:
qdata.tree()
qdata.browse()

bobleesj/quantem-data  (30.9 GB total, 11 datasets)

4dstem/   (7 datasets, 30.80 GB)
     3157.9 MB  gold_30mrad1.3mx04
     7081.7 MB  gold_30mrad1.3mx06
     7120.0 MB  gold_30mrad1.3mx07
     7045.0 MB  gold_30mrad1.3mx09
     4889.1 MB  gold_512
     1208.0 MB  gold_512_npy_bin4
      302.0 MB  gold_512_npy_bin8

haadf/   (4 datasets, 0.08 GB)
        8.7 MB  gold_drift_0deg
        8.7 MB  gold_drift_90deg
       33.8 MB  gold_haadf
       33.6 MB  gold_haadf_npy


## Load a real HAADF image

Load by dataset name. `quantem.data` downloads the file once into the Hugging Face cache, reads its calibration sidecar, and returns a ready-to-use `Dataset2d`. Future runs reuse the local cache.

In [3]:
full_dataset = qdata.load("gold_haadf_npy", verbose=False)
full_image = full_dataset.array
pixel_size_nm = float(full_dataset.sampling[0])

print(
    f"{full_dataset.name}: {full_image.shape[0]} x {full_image.shape[1]} "
    f"{full_image.dtype}, {pixel_size_nm:.4f} {full_dataset.units[0]}/pixel"
)

gold_haadf_npy: 4096 x 4096 uint16, 0.0186 nm/pixel


## Show2D preview

For docs and Colab startup speed, take a 512 by 512 preview from the full image. The sampling is scaled by the stride so the scale bar remains calibrated.

In [4]:
stride = 8
preview = np.asarray(full_image[::stride, ::stride], dtype=np.float32)
preview -= preview.min()
preview /= preview.max() or 1.0

preview_dataset = Dataset2d.from_array(
    preview,
    sampling=(pixel_size_nm * stride, pixel_size_nm * stride),
    units=tuple(full_dataset.units),
    name="Gold HAADF preview",
)

Show2D(preview_dataset, offline=True, cmap="inferno")

Show2D(512×512, cmap=inferno)

## Show3D from downloaded data

A time or depth stack can use the same download/cache pattern. For a lightweight demo, this cell makes a small aligned stack from the downloaded HAADF preview so `Show3D` has real image texture without downloading another large file.

In [5]:
shifts = np.linspace(-12, 12, 24).astype(int)
stack = np.stack([np.roll(preview, shift, axis=0) for shift in shifts]).astype(np.float32)
stack_dataset = Dataset3d.from_array(
    stack,
    sampling=(1.0, pixel_size_nm * stride, pixel_size_nm * stride),
    units=("frame", *tuple(full_dataset.units)),
    name="Gold HAADF preview stack",
)

Show3D(stack_dataset, offline=True, fps=12, cmap="inferno")

Show3D(24×512×512, frame=12, cmap=inferno)

## Larger files

The same API works for larger datasets; choose those intentionally because they can be hundreds of MB or several GB.

```python
qdata.tree()                         # see names and sizes
path = qdata.download("gold_512")    # raw cached folder, no widget load yet
# data = qdata.load("gold_512", det_bin=4)  # load when you really want the full 4D-STEM data
```

For shareable tutorials, use a small calibrated preview in the notebook and link to the full dataset page. For local analysis, load the full cached dataset and pass it directly into the widget.